In [ ]:
# RAILPULSE | NB_01_Source_Simulator  (lakehouse par défaut : LH_RailPulse_Bronze)
# Simule le système source de l'opérateur : 12 mois de données brutes déposées dans la zone d'atterrissage.
import os, json
import numpy as np
import pandas as pd
fr = lambda n: f"{n:,}".replace(",", " ")   # 98 950 plutôt que 98,950

try:   # garde-fou : les fichiers vont dans le lakehouse par défaut, il doit être Bronze
    _lh = notebookutils.runtime.context.get("defaultLakehouseName")
    if _lh and _lh != "LH_RailPulse_Bronze":
        raise ValueError(f"Lakehouse par défaut = {_lh}. Définissez LH_RailPulse_Bronze comme lakehouse par défaut.")
except NameError:
    pass
BASE = os.environ.get("RAILPULSE_BASE", "/lakehouse/default/Files/landing")
START, END = "2025-10-01", "2026-09-30"          # 12 mois d'exploitation
PLAN_START, PLAN_DAYS = "2026-10-21", 14         # horaire des 14 prochains jours (à prévoir par le ML)
rng = np.random.default_rng(2026)                # graine fixe : mêmes chiffres pour tout le monde

# ---------- 1. Gares ----------
stations = pd.DataFrame([
    ("PAR", "Paris Est",          "Paris",      "FR", 48.8768, 2.3592, 1),
    ("LUX", "Luxembourg",         "Luxembourg", "LU", 49.6000, 6.1339, 1),
    ("MTZ", "Metz Ville",         "Metz",       "FR", 49.1097, 6.1772, 0),
    ("STR", "Strasbourg",         "Strasbourg", "FR", 48.5850, 7.7350, 1),
    ("BRU", "Bruxelles Midi",     "Bruxelles",  "BE", 50.8357, 4.3360, 1),
    ("AMS", "Amsterdam Centraal", "Amsterdam",  "NL", 52.3791, 4.9003, 0),
    ("CGN", "Köln Hbf",           "Cologne",    "DE", 50.9430, 6.9589, 1),
    ("FRA", "Frankfurt Hbf",      "Francfort",  "DE", 50.1071, 8.6638, 1),
    ("BSL", "Basel SBB",          "Bâle",       "CH", 47.5476, 7.5897, 1),
    ("ZRH", "Zürich HB",          "Zurich",     "CH", 47.3779, 8.5403, 1),
    ("LYS", "Lyon Part-Dieu",     "Lyon",       "FR", 45.7606, 4.8594, 1),
    ("MIL", "Milano Centrale",    "Milan",      "IT", 45.4864, 9.2048, 0),
], columns=["station_code", "station_name", "city", "country_code", "latitude", "longitude", "is_hub"])

# ---------- 2. Lignes : (code, A, B, km, minutes, départs/jour/sens, sièges, prix 2de €/km) ----------
lines = [
    ("L01", "PAR", "LUX", 380, 125,  6, 510, 0.115), ("L02", "PAR", "STR", 490, 106, 10, 510, 0.118),
    ("L03", "PAR", "BRU", 315,  82, 12, 450, 0.140), ("L04", "BRU", "AMS", 210, 110, 10, 450, 0.120),
    ("L05", "BRU", "CGN", 220, 115,  7, 450, 0.115), ("L06", "CGN", "FRA", 180,  64, 14, 460, 0.110),
    ("L07", "LUX", "MTZ",  60,  45, 20, 300, 0.090), ("L08", "STR", "FRA", 220, 110,  6, 460, 0.105),
    ("L09", "STR", "BSL", 140,  80, 12, 300, 0.095), ("L10", "BSL", "ZRH",  87,  55, 16, 400, 0.120),
    ("L11", "PAR", "LYS", 430, 117, 14, 510, 0.112), ("L12", "ZRH", "MIL", 280, 197,  8, 420, 0.108),
]
ctry = dict(zip(stations.station_code, stations.country_code))
routes = []
for code, a, b, km, mins, n, seats, ppk in lines:
    for o, d in ((a, b), (b, a)):
        routes.append(dict(line_code=code, origin=o, destination=d, distance_km=km, duration_min=mins,
                           departures=n, seats=seats, price_per_km=ppk,
                           cross_border=int(ctry[o] != ctry[d])))
routes = pd.DataFrame(routes)

# ---------- 3. Parc de rames ----------
trainsets = pd.DataFrame({
    "trainset_id": [f"TS{i:02d}" for i in range(1, 41)],
    "series": rng.choice(["Velaro E", "Duplex X", "Regio R"], 40, p=[0.45, 0.40, 0.15]),
    "commissioning_year": rng.integers(2008, 2025, 40),
})

# ---------- 4. Météo quotidienne par gare (sévérité 0 à 3) ----------
days = pd.date_range(START, END, freq="D")
holidays = pd.to_datetime(["2025-11-01", "2025-11-11", "2025-12-25", "2025-12-26", "2026-01-01", "2026-04-06",
                           "2026-05-01", "2026-05-08", "2026-05-14", "2026-05-25", "2026-07-14", "2026-08-15"])

def weather_for(dates):
    rows = []
    for _, s in stations.iterrows():
        doy = dates.dayofyear.values
        temp = 11 - (s.latitude - 48) * 0.8 - 9 * np.cos(2 * np.pi * (doy - 20) / 365) + rng.normal(0, 3, len(dates))
        winter = (dates.month.isin([12, 1, 2])).astype(float)
        p = np.clip(0.04 + 0.10 * winter + 0.03 * (s.country_code == "CH"), 0, 1)
        sev = rng.binomial(3, p)
        rows.append(pd.DataFrame({"weather_date": dates.strftime("%Y-%m-%d"), "station_code": s.station_code,
                                  "temp_c": temp.round(1), "precip_mm": (rng.gamma(0.6, 3, len(dates)) * (1 + sev)).round(1),
                                  "wind_kmh": (rng.gamma(3, 6, len(dates)) + sev * 12).round(0), "severity": sev}))
    return pd.concat(rows, ignore_index=True)

weather = weather_for(days)

# ---------- 5. Circulations (une ligne = un train) ----------
def build_trips(dates, with_outcome=True):
    grid = []
    for r in routes.itertuples():
        hours = np.linspace(6.0, 21.0, r.departures)
        for h in hours:
            grid.append((r.Index, int(h), int(round((h % 1) * 60 / 5) * 5) % 60))
    g = pd.DataFrame(grid, columns=["ridx", "hh", "mm"])
    t = g.merge(pd.DataFrame({"service_date": dates}), how="cross").merge(routes, left_on="ridx", right_index=True)
    t["dep"] = t.service_date + pd.to_timedelta(t.hh, "h") + pd.to_timedelta(t.mm, "m")
    t["arr"] = t.dep + pd.to_timedelta(t.duration_min, "m")
    t["trip_id"] = (t.service_date.dt.strftime("%Y%m%d") + "-" + t.origin + t.destination + "-"
                    + t.hh.astype(str).str.zfill(2) + t.mm.astype(str).str.zfill(2))
    t["trainset_id"] = rng.choice(trainsets.trainset_id.values, len(t))
    w = weather.set_index(["weather_date", "station_code"]).severity if with_outcome else wplan
    key = t.service_date.dt.strftime("%Y-%m-%d")
    t["sev"] = np.maximum(w.reindex(list(zip(key, t.origin))).values, w.reindex(list(zip(key, t.destination))).values)
    age = 2026 - t.trainset_id.map(dict(zip(trainsets.trainset_id, trainsets.commissioning_year)))
    peak = t.hh.isin([7, 8, 17, 18]).astype(int)
    wkd = t.service_date.dt.dayofweek
    season = 1 + 0.18 * t.service_date.dt.month.isin([7, 8, 12]) + 0.10 * t.service_date.isin(holidays)
    load = np.clip(0.52 + 0.20 * peak + 0.10 * wkd.isin([4, 6]) - 0.08 * (wkd == 5)
                   + 0.06 * (t.distance_km > 300) + rng.normal(0, 0.09, len(t)), 0.15, 1.0) * season
    t["load_factor"] = np.clip(load, 0.15, 1.0)
    t["age"], t["peak"] = age, peak
    if not with_outcome:
        return t
    base = rng.exponential(1.6, len(t)) - 1.2
    delay = (base + peak * rng.uniform(0, 2.5, len(t)) + t.sev * rng.gamma(2.0, 2.0, len(t))
             + np.clip(age - 8, 0, None) * 0.22 + (t.load_factor > 0.9) * 2.0 + t.cross_border * 1.0)
    incident = rng.random(len(t)) < 0.018
    delay = delay + incident * rng.uniform(20, 95, len(t))
    t["arrival_delay_min"] = np.round(delay).astype(int)
    t["cancelled"] = rng.random(len(t)) < (0.006 + 0.012 * (t.sev == 3))
    return t

trips = build_trips(days)

# ---------- 6. Réservations (train x classe x canal) ----------
ok = trips[~trips.cancelled]
pax = np.round(ok.seats * ok.load_factor).astype(int)
channels, ch_p = np.array(["App", "Web", "Gare", "Partenaire"]), np.array([0.46, 0.31, 0.13, 0.10])
bk = []
for cls, share, mult in ((1, 0.18, 1.45), (2, 0.82, 1.00)):
    pc = rng.binomial(pax, share)
    split = np.array([rng.multinomial(n, ch_p) for n in pc])
    for j, ch in enumerate(channels):
        lead = np.clip(rng.normal(18 - 4 * j, 6, len(ok)), 0, 90)
        price = ok.distance_km.values * ok.price_per_km.values * mult * (0.75 + 0.6 * ok.load_factor.values) * (1.15 - lead / 200)
        bk.append(pd.DataFrame({"trip_id": ok.trip_id.values, "travel_class": cls, "channel": ch,
                                "passengers": split[:, j], "revenue_eur": np.round(split[:, j] * price, 2),
                                "avg_lead_days": lead.round(1)}))
bookings = pd.concat(bk, ignore_index=True)
bookings = bookings[bookings.passengers > 0]

# ---------- 7. Défauts volontaires (la couche Silver doit les intercepter) ----------
out = trips.assign(scheduled_departure=trips.dep.dt.strftime("%Y-%m-%dT%H:%M:%S"),
                   scheduled_arrival=trips.arr.dt.strftime("%Y-%m-%dT%H:%M:%S"),
                   service_date=trips.service_date.dt.strftime("%Y-%m-%d"),
                   status=np.where(trips.cancelled, "CANCELLED", "COMPLETED"),
                   arrival_delay_min=trips.arrival_delay_min.astype("Int64").where(~trips.cancelled))
cols = ["trip_id", "service_date", "line_code", "origin", "destination", "scheduled_departure",
        "scheduled_arrival", "trainset_id", "status", "arrival_delay_min", "seats", "distance_km"]
out = out[cols]
bad = rng.choice(len(out), 65, replace=False)
out.loc[out.index[bad[:40]], "arrival_delay_min"] = -999           # capteur défaillant
out.loc[out.index[bad[40:]], "origin"] = "XXX"                    # gare inconnue
dups = out.sample(n=400, random_state=7)                          # doublons de transmission
out = pd.concat([out, dups]).sort_values(["scheduled_departure", "trip_id"]).reset_index(drop=True)

# ---------- 8. Horaire des 14 prochains jours + météo prévue ----------
plan_days = pd.date_range(PLAN_START, periods=PLAN_DAYS, freq="D")
wplan_df = weather_for(plan_days)
wplan = wplan_df.set_index(["weather_date", "station_code"]).severity
plan = build_trips(plan_days, with_outcome=False)
plan_out = plan.assign(service_date=plan.service_date.dt.strftime("%Y-%m-%d"),
                       scheduled_departure=plan.dep.dt.strftime("%Y-%m-%dT%H:%M:%S"))[
    ["trip_id", "service_date", "line_code", "origin", "destination", "scheduled_departure", "trainset_id",
     "seats", "distance_km", "load_factor"]].round({"load_factor": 3})

# ---------- 9. Dépôt dans la zone d'atterrissage ----------
for sub in ["reference", "trips", "bookings", "plan"]:
    os.makedirs(f"{BASE}/{sub}", exist_ok=True)
stations.to_csv(f"{BASE}/reference/stations.csv", index=False)
pd.concat([weather, wplan_df]).to_csv(f"{BASE}/reference/weather_daily.csv", index=False)
with open(f"{BASE}/reference/trainsets.json", "w") as f:
    json.dump(trainsets.astype({"commissioning_year": int}).to_dict("records"), f)
month = out.service_date.str[:7]
for m in sorted(month.unique()):
    out[month == m].to_csv(f"{BASE}/trips/trips_{m}.csv", index=False)
bmonth = bookings.trip_id.str[:4] + "-" + bookings.trip_id.str[4:6]
for m in sorted(bmonth.unique()):
    bookings[bmonth == m].to_parquet(f"{BASE}/bookings/bookings_{m}.parquet", index=False)
plan_out.to_csv(f"{BASE}/plan/timetable_next14.csv", index=False)

# ---------- 10. Tables Bronze de référence (le Dataflow Gen2 les lira) ----------
wx_all = pd.concat([weather, wplan_df], ignore_index=True)
wx_all["weather_date"] = pd.to_datetime(wx_all.weather_date).dt.date
spark.createDataFrame(stations).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("LH_RailPulse_Bronze.bronze_stations")
spark.createDataFrame(wx_all).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("LH_RailPulse_Bronze.bronze_weather")

print(f"Gares {len(stations)} | lignes {len(lines)} | rames {len(trainsets)}")
print(f"Circulations déposées {fr(len(out))} (dont 400 doublons et 65 lignes défectueuses)")
print(f"Lignes de réservation {fr(len(bookings))} | horaire à prévoir {fr(len(plan_out))} trains")